In [1]:
import sys
sys.path.append("../")
from py_code import DATroot_events_reader as dat
import pandas as pd
import pyarrow as pa, pyarrow.parquet as pq, gc
import glob
import os

# Extract The Traces from the ROOT files from the HAWCSim Simulation

Root files are kept in a storage directory separated by array type, energy and zenith angle range. In the following snippets traces are read for all the stations event by event.

- Specify saving paths for the parquet of the traces.
- Specify time window of acquisition from the trigger time in the shower plane.
- Select minimum and maximum reconstructed (linear fit on total nPE) energy range.
- Select maximum $\theta$ inclination.
- Set the minimum and maximum DAT number for the files.
- **Create all required directories if they are not already present. Label each directory clearly and unambiguously, using descriptive names that reflect its purpose and contents.**
- **Make sure that Train and Test have different DAT ranges for protons.**
- **Make sure all other parameters are set to the same value to avoid dataset mismatches.**

In [2]:
# Specify path for the protons root files
DAT_path_protons = "/swgo/Array_Simulato_Analisi_ML/roots_mPMT_protons_SWGO_array/4FF_R600m_h175r180_black_uniform0600m_100300TeV_030deg/"

# Specify path for the gammas root files
DAT_path_gammas = "/swgo/Array_Simulato_Analisi_ML/roots_mPMT_gammas_SWGO_array/4FF_R600m_h175r180_black_uniform0600m_100300TeV_030deg/"


# Set ranges
time_window = 500        #ns
min_energy_reco = 100e3  #GeV
max_energy_reco = 160e3  #GeV
max_theta = 25           #deg

TRAIN_DAT_MIN = 0
TRAIN_DAT_MAX = 100               #numbers above 9999 will break the code

TEST_PROTONS_DAT_MIN = 101
TEST_PROTONS_DAT_MAX = 5500       #numbers above 9999 will break the code

TEST_GAMMAS_DAT_MIN = 0
TEST_GAMMAS_DAT_MAX = 600         #numbers above 9999 will break the code

# TRAIN TRACES (Protons)

In [3]:
# Specify Directory for single muon stations traces
smu_path = "../dfs_traces/HAWCSim_array/train/sm/100160TeV_025deg_4FF_180R175h_Black/13petrain_13petest_500ns/"  

# Specify Directory for non-muon stations traces
nmu_path = "../dfs_traces/HAWCSim_array/train/bkg/100160TeV_025deg_4FF_180R175h_Black/13petrain_13petest_500ns/"

# Clean such directories if they already exist 
for f in glob.glob(smu_path + "*.parquet"):
    os.remove(f)

for f in glob.glob(nmu_path + "*.parquet"):
    os.remove(f)

# THIS IS A REMINDER OF THE FIT THAT IS BEING APPLIED FOR ENERGY RECONSTRUCTION, PLEASE CHANGE IT AS YOU CHANGE THE FIT. 
# CURRENT FIT : "H4FF"

# Prepare Train traces
for i in range(TRAIN_DAT_MIN, TRAIN_DAT_MAX):

    DAT_name = f"DAT140{i:04d}"
    DAT_file_name = f"{DAT_name}.root"

    p_train = dat.Read_Roots_For_Training(
                                        output_dir_smu=smu_path,
                                        output_dir_nmu=nmu_path,
                                        DATname=DAT_name,
                                        path_to_DAT_root=DAT_path_protons + DAT_file_name,
                                        fit_type="H4FF",
                                        max_time=time_window,
                                        min_energy=min_energy_reco,
                                        max_energy=max_energy_reco,
                                        max_theta=max_theta
                                        )

Working on  /swgo/Array_Simulato_Analisi_ML/roots_mPMT_protons_SWGO_array/4FF_R600m_h175r180_black_uniform0600m_100300TeV_030deg/DAT1400010.root
Number of primaries in this DAT  27
E_reco [TeV] =  124.02557746883645
E_reco [TeV] =  122.44674354276682
E_reco [TeV] =  108.23364586701153
Primaries after the cut =  3
Working on  /swgo/Array_Simulato_Analisi_ML/roots_mPMT_protons_SWGO_array/4FF_R600m_h175r180_black_uniform0600m_100300TeV_030deg/DAT1400012.root
Number of primaries in this DAT  23
E_reco [TeV] =  116.76006753601322
E_reco [TeV] =  101.90034845708948
E_reco [TeV] =  114.40708409670582
E_reco [TeV] =  152.3691489743866
E_reco [TeV] =  139.9396486690376
E_reco [TeV] =  111.93375722958652
Primaries after the cut =  6
Working on  /swgo/Array_Simulato_Analisi_ML/roots_mPMT_protons_SWGO_array/4FF_R600m_h175r180_black_uniform0600m_100300TeV_030deg/DAT1400013.root
Number of primaries in this DAT  19
E_reco [TeV] =  122.77005424435104
E_reco [TeV] =  139.22836512555233
E_reco [TeV] =  

# TEST PROTONS TRACES

In [5]:
test_protons_path = "../dfs_traces/HAWCSim_array/test_protons/100160TeV_025deg_4FF_180R175h_Black/13petrain_13petest_500ns/"

for f in glob.glob(test_protons_path + "*.parquet"):
    os.remove(f)

# THIS IS A REMINDER OF THE FIT THAT IS BEING APPLIED FOR ENERGY RECONSTRUCTION, PLEASE CHANGE IT AS YOU CHANGE THE FIT. 
# CURRENT FIT : "H4FF"

# Prepare Protons Test traces
for i in range(TEST_PROTONS_DAT_MIN, TEST_PROTONS_DAT_MAX):

    DAT_name = f"DAT140{i:04d}"
    DAT_file_name = f"{DAT_name}.root"
    
    p_test_protons = dat.Read_Roots_For_Testing(
                                               output_dir = test_protons_path, 
                                               DATname = DAT_name, 
                                               path_to_DAT_root = DAT_path_protons + DAT_file_name, 
                                               fit_type = "H4FF", 
                                               max_time = time_window, 
                                               min_energy = min_energy_reco, 
                                               max_energy = max_energy_reco, 
                                               max_theta = max_theta
                                               )

Working on  /swgo/Array_Simulato_Analisi_ML/roots_mPMT_protons_SWGO_array/4FF_R600m_h175r180_black_uniform0600m_100300TeV_030deg/DAT1400103.root
Number of primaries in this DAT  23
E_reco [TeV] =  110.69619571074469
E_reco [TeV] =  110.42497395552681
E_reco [TeV] =  127.70952329633222
E_reco [TeV] =  133.81829938570968
E_reco [TeV] =  112.71688759564609
E_reco [TeV] =  129.9637173546
E_reco [TeV] =  143.45116212235513
E_reco [TeV] =  141.30294212738443
Primaries after the cut =  8
All Done ! :))
Working on  /swgo/Array_Simulato_Analisi_ML/roots_mPMT_protons_SWGO_array/4FF_R600m_h175r180_black_uniform0600m_100300TeV_030deg/DAT1400104.root
Number of primaries in this DAT  23
E_reco [TeV] =  114.20591299349786
E_reco [TeV] =  127.12576786291626
E_reco [TeV] =  114.39630707331968
Primaries after the cut =  3
All Done ! :))
Working on  /swgo/Array_Simulato_Analisi_ML/roots_mPMT_protons_SWGO_array/4FF_R600m_h175r180_black_uniform0600m_100300TeV_030deg/DAT1400105.root
Number of primaries in t

# TEST GAMMAS TRACES

In [7]:
test_gammas_path = "../dfs_traces/HAWCSim_array/test_gammas/100160TeV_025deg_4FF_180R175h_Black/13petrain_13petest_500ns/"

for f in glob.glob(test_gammas_path + "*.parquet"):
    os.remove(f)

# THIS IS A REMINDER OF THE FIT THAT IS BEING APPLIED FOR ENERGY RECONSTRUCTION, PLEASE CHANGE IT AS YOU CHANGE THE FIT. 
# CURRENT FIT : "H4FF"

# Prepare Gammas Test traces
for i in range(TEST_GAMMAS_DAT_MIN, TEST_GAMMAS_DAT_MAX):

    DAT_name = f"DAT10{i:04d}"
    DAT_file_name = f"{DAT_name}.root"
    
    p_test_gammas = dat.Read_Roots_For_Testing(
                                       output_dir = test_gammas_path, 
                                       DATname = DAT_name, 
                                       path_to_DAT_root = DAT_path_gammas + DAT_file_name, 
                                       fit_type = "H4FF", 
                                       max_time = time_window, 
                                       min_energy = min_energy_reco, 
                                       max_energy = max_energy_reco, 
                                       max_theta = max_theta
                                       )

Working on  /swgo/Array_Simulato_Analisi_ML/roots_mPMT_gammas_SWGO_array/4FF_R600m_h175r180_black_uniform0600m_100300TeV_030deg/DAT100000.root
Number of primaries in this DAT  7
E_reco [TeV] =  102.62600136508962
E_reco [TeV] =  122.46470524841038
E_reco [TeV] =  129.88648202033264
E_reco [TeV] =  157.18108991629845
E_reco [TeV] =  104.07910335165427
Primaries after the cut =  5
All Done ! :))
Working on  /swgo/Array_Simulato_Analisi_ML/roots_mPMT_gammas_SWGO_array/4FF_R600m_h175r180_black_uniform0600m_100300TeV_030deg/DAT100001.root
Number of primaries in this DAT  8
E_reco [TeV] =  145.6802097927219
E_reco [TeV] =  118.8346445378453
Primaries after the cut =  2
All Done ! :))
Working on  /swgo/Array_Simulato_Analisi_ML/roots_mPMT_gammas_SWGO_array/4FF_R600m_h175r180_black_uniform0600m_100300TeV_030deg/DAT100002.root
Number of primaries in this DAT  7
E_reco [TeV] =  132.1783956604519
Primaries after the cut =  1
All Done ! :))
Working on  /swgo/Array_Simulato_Analisi_ML/roots_mPMT_ga